In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from datetime import datetime as dt, timedelta
from tiingo import TiingoClient
import requests

# Load API keys from a local secrets file (kept out of version control via .gitignore)
try:
    _secrets_dir = os.path.dirname(os.path.abspath(__file__))  # works when run as a .py script
except NameError:
    _secrets_dir = os.getcwd()  # fallback for notebooks, where __file__ is not defined

with open(os.path.join(_secrets_dir, "secrets.json")) as f:
    secrets = json.load(f)

config = {
    "api_key": secrets["tiingo_api_key"]
}
client = TiingoClient(config)

api_key2 = secrets["polygon_api_key"]


In [ ]:
# Directory where this notebook/script lives, so files load regardless of the current working directory
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))  # works when run as a .py script
except NameError:
    SCRIPT_DIR = os.getcwd()  # fallback for notebooks, where __file__ is not defined

def load_data(file_path):
    return pd.read_excel(os.path.join(SCRIPT_DIR, file_path), index_col=0, parse_dates=True)

bond1 = 'ZFL.TO.xlsx'
bond2 = 'ZFM.TO.xlsx'
equity = 'VFV.TO.xlsx'
gold = 'CGL.TO.xlsx'

bond1_df = load_data(bond1)
bond2_df = load_data(bond2)
equity_df = load_data(equity)
gold_df = load_data(gold)

print("Bond Long:", bond1_df.head())
print("Bond Mid:", bond2_df.head())
print("Equity (VFV.TO)", equity_df.head())
print("Gold (CGL.TO)", gold_df.head())

In [ ]:
# Data cleaning
def clean_data(df):
    df = df.dropna().copy()
    df['Returns'] = np.log(df['Close'] / df['Close'].shift(1)) #Computing Log returns (vectorized)
    return df

dataframes = {'bond1': bond1_df, 'bond2': bond2_df, 'equity': equity_df, 'gold': gold_df}

dataframes = {key: clean_data(df) for key, df in dataframes.items()}

In [ ]:
# Plotting historical prices for all assets
plt.figure(figsize=(12,6))
for key, df in dataframes.items():
    plt.plot(df.index, df['Close'], label=key)
plt.xlabel('Date')
plt.ylabel('Price')
plt.title('Historical Price Data')
plt.legend()
plt.show()

In [ ]:
#Compute Volatility
def comp_vol(df, window=30):
    return df['Returns'].rolling(window).std()*np.log(252)

for key in dataframes:
    dataframes[key].loc[:, 'Volatility'] = comp_vol(dataframes[key])

print(dataframes)

In [ ]:
# Plotting volatility
plt.figure(figsize=(12,6))
for key, df in dataframes.items():
    plt.plot(df.index, df['Volatility'], label=key)
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.title('Historical Volatility Data')
plt.legend()
plt.show()

In [ ]:
#Cleaning Tbills data
dfT = pd.read_excel("tbill_3mnth.xlsx")
dfT.columns = ['date', 'rf']
dfT['date'] = pd.to_datetime(dfT['date'], errors='coerce')
dfT['rf'] = pd.to_numeric(dfT['rf'], errors='coerce')
dfT = dfT.dropna(subset=['rf'])
dfT['rf'] = dfT['rf']/100
dfT['rf_daily'] = (1+dfT['rf'])**(1/252) - 1
dfT['rf_log_daily'] = np.log(1+dfT['rf_daily'])
dfT


In [ ]:
# Getting market data for option variables
symbol = "NVDA"
df = client.get_dataframe(symbol, startDate='2020-01-01', endDate=dt.today() - timedelta(days=1))
df = df.sort_index(ascending=False)   # Most recent date first
print(df)

log_returns = np.log(df['adjClose']/df['adjClose'].shift(1))
S = df['close'].iloc[1]  # Last Close Price
print(S)
opt_exp = dt(2026, 7, 31)   # Expiration date (matches the option chain queried below)
T = (opt_exp - dt.now()).total_seconds() / (365 * 24 * 60 * 60)    # Time till expiration (in years)
print(T)
r = dfT['rf_log_daily'].iloc[1]    # Risk-free rate using 3 month Tbills
print(r)
sigma = log_returns.std()*np.sqrt(252)  # Volatility of underlying asset
print(sigma)

params = {
    "underlying_ticker": symbol,
    "contract_type": "call",
    "expiration_date": "2026-07-31",
    "strike_price.gte": round(S * 0.9, 0),  
    "strike_price.lte": round(S * 1.15, 0),   
    "expired": "false",
    "sort": "strike_price",
    "order": "asc",
    "limit": 250,
    "apiKey": api_key2,
}


url = "https://api.polygon.io/v3/reference/options/contracts"
resp = requests.get(url, params=params, timeout=15)
resp.raise_for_status()
res = resp.json()

print(res)   # inspect once while debugging

if "results" not in res:
    raise ValueError(f"Polygon response missing 'results': {res}")

if not res["results"]:
    raise ValueError(f"No option contracts found for params={params}")

K = [c["strike_price"] for c in res["results"]]
S_underlying = S
print(K)


In [ ]:
# Probability/Conditional Functions
def N(d):
    return norm.cdf(d)

def N_prime(d):
    return norm.pdf(d)

In [ ]:
# Black-Scholes option pricing model (A Physics model with no friction, i.e. it prices options under ideal conditions)
def black_scholes(S, K, T, r, sigma, option_type='call'):
    if T <= 0:
        # Option has expired: value equals intrinsic value (no time value left)
        if option_type == 'call':
            return max(S - K, 0.0)
        elif option_type == 'put':
            return max(K - S, 0.0)
        else:
            raise ValueError("Invalid option type. Must be 'call' or 'put'.")

    d1 = (np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if option_type == 'call':
        return S*N(d1) - K*(np.e**((-r)*T))*N(d2)
    elif option_type == 'put':
        return K*np.e**(-r*T) * N(-d2) - S*N(-d1)
    else:
        raise ValueError("Invalid option type. Must be 'call' or 'put'.")
    
# Market > B-S => opt is expensive, Market < B-S => opt is cheap
# Traders also exctract implied volatility from B-S !!DO THIS NEXT

blackScholes = pd.DataFrame({
    "Strike Price": K,
    "B-S Call": [black_scholes(S,strike,T,r,sigma,'call') for strike in K],
    "B-S Put": [black_scholes(S,strike,T,r,sigma,'put') for strike in K]
})
display(blackScholes)


In [ ]:
# Calulcating Greeks
d1 = (np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)

# Delta
def delta(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*np.sqrt(T))
    if option_type=='call':
        return N(d1)
    elif option_type=='put':
        return N(d1) - 1

# Gamma
def gamma(S,K,T,r,sigma):
    d1 = (np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*np.sqrt(T))
    return N_prime(d1)/(S*sigma*np.sqrt(T))

# Theta
def theta(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if option_type == 'call':
        return ((-S*N_prime(d1)*sigma)/(2*np.sqrt(T))) - (r*K*(np.e**(-r*T))*N(d2))
    elif option_type == 'put':
        return ((-S*N_prime(d1)*sigma)/(2*np.sqrt(T))) + (r*K*(np.e**(-r*T))*N(-d2))
    else:
        print("Option type must be 'put' or 'call'")

# Vega
def vega(S, K, T, r, sigma):
    d1 = (np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*np.sqrt(T))
    return S*N_prime(d1)*np.sqrt(T)

# Rho
def rho(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if option_type == 'call':
        return K*T*(np.e**(-r*T))*N(d2)
    elif option_type == 'put':
        return -K*T*(np.e**(-r*T))*N(-d2)
    else:
        print("Option type must be 'put' or 'call'")

In [ ]:
# Greeks Calculations for given stock

def build_greeks(option_type):
    return pd.DataFrame({
    "Strike Price": K,
    "Delta": [delta(S, strike, T, r, sigma, option_type) for strike in K],
    "Gamma": [gamma(S, strike, T, r, sigma) for strike in K],
    "Theta": [theta(S, strike, T, r, sigma, option_type) for strike in K],
    "Vega":  [vega(S, strike, T, r, sigma) for strike in K],
    "Rho":   [rho(S, strike, T, r, sigma, option_type) for strike in K],
})

greeks_put = build_greeks('put')
greeks_call = build_greeks('call')

print(f"Greeks for {symbol} (call)")
display(greeks_call)

print(f"Greeks for {symbol} (put)")
display(greeks_put)


In [ ]:
# Monte Carlo Pricing Model
# Simulate terminal prices once; the strike only enters the payoff, not the simulation.
def simulate_terminal(S, r, sigma, T, N, seed=None):
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(N)
    return S*np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)

n_sims = 10000000
St = simulate_terminal(S, r, sigma, T, n_sims)   # one simulation, reused for all strikes
disc = np.exp(-r*T)

mc = pd.DataFrame({
    "Strike Price": K,
    "MC Call": [disc * np.maximum(St - strike, 0).mean() for strike in K],
    "MC Put":  [disc * np.maximum(strike - St, 0).mean() for strike in K],
})
display(mc)

In [ ]:
# Comparing Monte Carlo and Black-Scholes models
print("Monte Carlo and Black-Scholes comparison")
mc_bs = pd.DataFrame({
    "Strike Price": K,
    "MC Call":  [disc * np.maximum(St - strike, 0).mean() for strike in K],
    "B-S Call": [black_scholes(S, strike, T, r, sigma, 'call') for strike in K],
    "MC Put":   [disc * np.maximum(strike - St, 0).mean() for strike in K],
    "B-S Put":  [black_scholes(S, strike, T, r, sigma, 'put') for strike in K],
})

mc_bs["Call Diff"] = (mc_bs["MC Call"] - mc_bs["B-S Call"]).abs()
mc_bs["Put Diff"]  = (mc_bs["MC Put"]  - mc_bs["B-S Put"]).abs()

# Optional: reorder columns
mc_bs = mc_bs[["Strike Price", "MC Call", "B-S Call", "Call Diff",
               "MC Put", "B-S Put", "Put Diff"]]
display(mc_bs)

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(mc_bs["Strike Price"], mc_bs["Call Diff"], 'o-', label="|MC - B-S| Call")
plt.plot(mc_bs["Strike Price"], mc_bs["Put Diff"],  'o-', label="|MC - B-S| Put")
plt.axhline(0, color='k', lw=0.8)
plt.xlabel("Strike Price"); plt.ylabel("Absolute Pricing Error")
plt.title(f"{symbol}: Monte Carlo vs Black-Scholes error by strike")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Monte Carlo convergence: pricing error vs number of simulations
Ns = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]

# Study one strike (ATM = closest to spot)
strike = min(K, key=lambda k: abs(k - S))
bs = black_scholes(S, strike, T, r, sigma, 'call')   # exact reference value

errors = []
for n in Ns:
    St_n = simulate_terminal(S, r, sigma, T, n)
    mc = np.exp(-r*T) * np.maximum(St_n - strike, 0).mean()
    errors.append(abs(mc - bs))

plt.figure(figsize=(10, 6))
plt.loglog(Ns, errors, 'o-', label="|MC - B-S|")
# Theoretical 1/sqrt(N) reference, anchored at the first point
plt.loglog(Ns, errors[0]*np.sqrt(Ns[0]/np.array(Ns)), 'k--', label=r"$\propto 1/\sqrt{N}$")
plt.xlabel("Number of simulations (N)")
plt.ylabel("Absolute pricing error")
plt.title(f"{symbol}: Monte Carlo convergence (strike={strike:.0f})")
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.show()

In [ ]:
# Implied Volatility Algorithm
# B-S: C = f(sigma), C = B-S price, f(sigma) is, smooth continous and strictly inc in sigma
# Conceptually finding f(sigma) - Market Price = 0

# MC: Conceptually solving MC Price(sigma) ± error = Market Price 

